# Глава 17. Токены, эмбеддинги и внимание

_Ноутбук сгенерирован из учебника «Машинное обучение для начинающих». Запускайте ячейки по порядку сверху вниз в Google Colab._

**Цель главы  🎯**

После этой главы ты сможешь объяснить, как текст превращается в числа (токены, эмбеддинги), описать идею механизма внимания своими словами и объяснить, почему языковые модели иногда галлюцинируют.

ChatGPT, Claude и другие чат-боты кажутся волшебством: они пишут тексты, отвечают на вопросы, сочиняют код. Но под капотом у них — большая языковая модель (LLM, Large Language Model), и её основную идею можно понять без единой формулы. В этой главе разберём три ключевых понятия — токены, эмбеддинги, внимание — и заодно поймём, почему модель иногда уверенно говорит неправду.

**Аналогия: очень умный автодополнитель  💡**

В основе LLM лежит поразительно простая задача: предсказать следующее слово. Как автодополнение в клавиатуре телефона, только несравнимо мощнее. «Небо сегодня…» → «голубое». Модель прочитала гигантские объёмы текста (по сути, значительную часть интернета) и научилась угадывать, что идёт дальше. Всё, что делает чат-бот, — это раз за разом предсказывает следующее слово, приписывая его к тексту. Из этого простого механизма, повторённого миллионы раз, и рождается связный осмысленный ответ.

Это стоит осознать: модель не «думает» в человеческом смысле и не хранит базу фактов. Она статистически предсказывает продолжение текста. Удивительно, что из такой простой идеи вырастает способность рассуждать, переводить и программировать — но именно из неё же растут и ограничения, о которых поговорим в конце главы.

## Шаг 1. Токены — кусочки текста

Модель не работает с буквами или целыми словами напрямую. Сначала текст режется на токены — кусочки, обычно чуть меньше слова. Например, «Машинное обучение» может стать токенами «Машин», «ное», «·обуч», «ение». Каждому возможному токену присвоен свой номер в словаре модели. Так текст превращается в последовательность чисел, с которой уже умеет работать нейросеть.

**Полезно знать  ℹ️**

Именно поэтому у моделей есть «лимит токенов», а не слов, и поэтому за длинные запросы к платному API берут больше — стоимость считается по числу токенов на входе и выходе. Грубый ориентир: 1000 токенов — это примерно 750 английских слов (для русского токенов на тот же текст обычно больше). Знать это полезно, когда будешь считать бюджет своего приложения.

## Шаг 2. Эмбеддинги — смысл в виде чисел

**Аналогия: карта смыслов  💡**

Эмбеддинг (embedding) — это способ записать смысл слова набором чисел, координатами на огромной «карте смыслов». На этой карте похожие по смыслу слова стоят рядом: «кот», «кошка», «котёнок» — в одном районе, а «автомобиль», «машина», «грузовик» — в другом, далёком. Модель понимает не буквы, а расположение слов на этой карте. Близкие числа означают близкий смысл — так модель «чувствует», что слова связаны, даже если написаны по-разному.

Из-за такой геометрии смыслов возможны почти магические вещи. Классический пример: если взять координаты слова «король», вычесть «мужчину» и прибавить «женщину», получится точка рядом со словом «королева». Модель улавливает не сами слова, а отношения между смыслами — «правитель мужского пола» минус «мужское» плюс «женское» = «правитель женского пола». Это упрощённая картинка (так чисто работали ранние эмбеддинги вроде word2vec; в современных контекстных моделях смысл слова зависит ещё и от соседних слов), но идея верна: смысл кодируется координатами, а близость координат означает близость смысла. Эмбеддинги — фундамент, на котором держится понимание языка.

## Шаг 3. Внимание — на какие слова смотреть

**Аналогия: на какие слова обращать внимание  💡**

В предложении «Кошка не поймала мышь, потому что она была слишком быстрой» — кто «она»? Кошка или мышь? Человек понимает по смыслу: раз не поймала, значит «быстрой» была мышь. Механизм внимания (attention) позволяет модели при обработке каждого слова смотреть на другие слова предложения и решать, какие из них сейчас важны. Слово «она» «обращает внимание» на «мышь» сильнее, чем на «кошку». Это и есть сердце современных языковых моделей.

_Архитектура, построенная вокруг механизма внимания, называется трансформер (transformer). Именно она стоит за буквой T в названии «ChatGPT» (Generative Pretrained Transformer). Внимание позволило моделям понимать длинный связный текст целиком, а не только пару соседних слов, — и именно этот прорыв 2017 года запустил всю нынешнюю революцию LLM._

## Посмотри на внимание своими глазами: BertViz

Внимание можно не только вообразить, но и увидеть. Инструмент BertViz рисует линии между словами: чем ярче и толще линия, тем сильнее одно слово «смотрит» на другое. Запусти в Colab и попробуй разные предложения — увидишь своими глазами, как местоимения тянутся к своим существительным, а прилагательные — к тем словам, которые описывают.

In [ ]:
# Запускать в Google Colab (нужен интернет для загрузки модели)
!pip install bertviz transformers -q

from transformers import AutoTokenizer, AutoModel
from bertviz import head_view

name = "bert-base-uncased"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModel.from_pretrained(name, output_attentions=True)

text = "The cat did not catch the mouse because it was fast"
inputs = tok(text, return_tensors="pt")
out = model(**inputs)

# Рисуем интерактивную карту внимания между словами
tokens = tok.convert_ids_to_tokens(inputs["input_ids"][0])
head_view(out.attentions, tokens)

_Ожидаемый результат: интерактивная схема связей между словами предложения._

## Почему LLM галлюцинируют

**Аналогия: уверенный отличник, который не знает  ⚠️**

Вспомни главную идею: модель всего лишь предсказывает правдоподобное следующее слово. Она не сверяется с базой фактов — она генерирует текст, который звучит убедительно и складно. Поэтому LLM может уверенно выдумать несуществующую книгу, неверную дату или фальшивую цитату. Это называется галлюцинацией. Модель — как отличник, который на любой вопрос отвечает гладко и уверенно, даже когда не знает ответа: для него главное — чтобы звучало правильно, а не чтобы было правдой.

Из этого следует главное правило работы с LLM: никогда не доверяй фактам от модели слепо. Проверяй важные цифры, имена, ссылки, даты по надёжным источникам. Модель прекрасно формулирует, переписывает, придумывает идеи, структурирует мысли — но она не является источником истины. Понимание этого ограничения — признак грамотного пользователя, который отличает мощный инструмент от волшебной палочки.

**Полезно знать  ℹ️**

Галлюцинации — не «баг», который скоро починят, а следствие самого принципа работы модели. Их можно уменьшить (например, дав модели проверенный текст — см. RAG в следующей главе), но полностью убрать нельзя. Поэтому критическое мышление при работе с ИИ будет нужно всегда.

## Задания

Базовый уровень

Открой любой онлайн-токенизатор (или спроси у ChatGPT/Claude) и посмотри, на сколько токенов разбивается фраза на русском и та же фраза на английском. Где токенов больше? Как это влияет на стоимость запроса?

Со звёздочкой  ⭐

Запусти BertViz на двух предложениях, где одно и то же местоимение относится к разным словам («Кошка гналась за мышью, потому что она была голодной» vs «…потому что она была быстрой»). Меняется ли, на какое слово смотрит «она»? Опиши наблюдение.

Вопросы для обсуждения  💬

Если модель просто предсказывает следующее слово, откуда берётся способность рассуждать и переводить?

Почему похожие по смыслу слова оказываются «рядом» на карте эмбеддингов? Чем это полезно?

Можно ли полностью избавить LLM от галлюцинаций? Почему да или почему нет?

**Проверь себя  ℹ️**

1. Какую базовую задачу решает языковая модель?

2. Что такое эмбеддинг и почему похожие слова оказываются рядом?

3. Почему модель может уверенно выдать ложный факт?